<h1> Block for imports <h1>

In [ ]:
# import block 
from flask import Flask, request, render_template, redirect, session, jsonify
from flask_session import Session 
from flask_socketio import emit, join_room, leave_room, SocketIO
from game import Room, Game, War  
import sqlite3
from werkzeug.security import generate_password_hash, check_password_hash
import numpy
import traceback
import random
import threading
import time
from static.states import fsm, state, blackjack_states, crazyeights_states

<h1> Global Variables Block <h1>

In [ ]:
# global variables block 
rooms = {}
socketio_connected = {} 
socketio_rooms = []                                        # WARNING: NO GARBAGE COLLECTION - POTENTIAL MEMORY LEAK 
sid_player_obj_mapping = {}                                # sid->player_obj, required to handle leaving rooms
app = Flask(__name__)
app.secret_key = "jack_of_all_games_secret_key"
app.config['SECRET_KEY'] = 'jack_of_all_games_secret_key'  
app.config['SESSION_TYPE'] = 'filesystem'                  # sessions stored server side 
app.config['SESSION_PERMANENT'] = True                     # persistent sessions even after browser closes 
app.config['PERMANENT_SESSION_LIFETIME'] = 3600            # keep session on server for x seconds  
Session(app)
socketio = SocketIO(app)

<h1>Database Block</h1>

In [ ]:
# database block 
def db():
    # best way to operate on db with sqlite
    conn = sqlite3.connect("../users.db")
    conn.row_factory = sqlite3.Row
    return conn

with db() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            wins INTEGER NOT NULL DEFAULT 0,
            games INTEGER NOT NULL DEFAULT 0
        )
    """)
    # creates user table if it doesnt exist

def record_win(username):
    with db() as conn:
        conn.execute("UPDATE users SET games = games + 1 WHERE username = ?", (username,))
        conn.execute("UPDATE users SET wins = wins + 1 WHERE username = ?", (username,))
    
def record_loss(username):
    with db() as conn:
        conn.execute("UPDATE users SET games = games + 1 WHERE username = ?", (username,))

<h1>Flask Homepage</h1>

In [ ]:
# flask homepage route 

@app.route('/')
def home_page():
    server_ip = request.host_url.rstrip("/")

    if "user_id" not in session:
        return redirect("/login")
    
    room_id = session.get("room")
    if room_id and room_id in rooms:
        return redirect("/room")
    else:
        session.pop("room", None)
        session.pop("player_index", None)
        session.pop("player_obj", None)
    return render_template("index.html", username=session.get("username"))

<h1>Registration</h1>

In [ ]:
# flask register route 

@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        # password validation (can add more strict requirements later)
        if len(username) < 3 or len(password) < 8:
            return "Username or password is invalid length", 400
        
        #generate hash and register to DB
        pw_hash = generate_password_hash(password)
        try:
            with db() as conn:
                conn.execute(
                    "INSERT INTO users (username, password_hash) VALUES (?, ?)",
                    (username, pw_hash)
                )
        
        # except if user exists alredy
        except sqlite3.IntegrityError:
            return "User already exists", 400
        
        return "", 200
    
    return render_template("register.html")

<h1>Login and Logout</h1>

In [ ]:
# flask login and logout 

@app.route("/login", methods=["GET", "POST"])
def login():
    if "user_id" in session:
        return redirect("/")
    
    if request.method == "POST": 
        username = request.form["username"]
        password = request.form["password"]

        # get user and hash from db
        with db() as conn:
            user = conn.execute(
                "SELECT * FROM users WHERE username = ?",
                (username,)
            ).fetchone()

            # if user doesnt exist or wrong password
            if not user or not check_password_hash(user["password_hash"], password):
                return "Invalid username or password", 401
            
        session["user_id"] = user["id"]
        session["username"] = user["username"]
        # retain info for session and redirect to game
        return "", 200
    
    return render_template("login.html")

@app.route("/logout")
def logout():
    # clear session data on logout
    session.clear()
    return redirect("/login")

<h1>Room Creation</h1>

In [ ]:
# room creation 

@app.route("/create_room", methods=["POST", "GET"])    
def create_room():                   # frontend sends a POST request and we make a room and auto join 
    # assumed frontend data format: 
    # form data with attributes: game_type, num_players 
    # frontend people, pls follow register.html example of sending form data 
    try: 
        data = request.form 
        game_type = data["game_type"]
        if not game_type in ["war", "blackjack", "crazyeights"]: 
            return "Cannot create room - game type not supported"
        num_players = int(data["num_players"])
        room = Room(socketio, game_type, num_players)
        rooms[room.id] = room                # store rooms in a dict for quick access 
        return room.id
    except:
        return "error, something went wrong. source: creating a room" 

<h1>Search Room</h1>

In [ ]:
# search room 

@app.route("/search_rooms", methods=["POST", "GET"])
def search_rooms(): 
    #print([room for room in rooms])
    data = {
        room.id: {
            "type": room.game_type, 
            "max_players": room.num_players, 
            "current_players": room.player_count,
        }

        for room in [rooms[_key] for _key in rooms.keys()]
    }
    return jsonify(data)

<h1>Room Handler</h1>

In [ ]:
# room handler 

@app.route("/room")
def frontend_room():
    if "room" not in session: return "You are not in a room"
    room_id = session["room"]

    if room_id not in rooms: return "Room does not exist"
    room = rooms[room_id]

    # Handle game types
    if room.game_type == "war":
        return render_template("war.html", code=room_id)

    if room.game_type == "blackjack":
        return render_template("blackjack.html", code=room_id)
    
    if room.game_type == "crazyeights":
        return render_template("crazyeights.html", code=room_id)

    if room.game_type == "poker":
        pass  # extension example

    return "Unsupported game type"

<h1>Leave a Room</h1>

In [ ]:
@app.route("/leave_room")
def backend_leaveroom():
    del session['room']
    print("Left a room")

<h1>Get Wins and Get Game</h1>

In [ ]:
@app.route("/get_wins")
def get_wins():
    with db() as conn:
        cur = conn.cursor()
        cur.execute("""SELECT username, wins FROM users""")
        rows = cur.fetchall()
        wins = [[row[0], row[1]] for row in rows]
        return wins

@app.route("/get_games")
def get_games():
    with db() as conn:
        cur = conn.cursor()
        cur.execute("""SELECT username, games FROM users""")
        rows = cur.fetchall()
        games = [[row[0], row[1]] for row in rows]
        return games

<h1>Socketio Validation</h1>

In [ ]:
def socket_validate(session):
    room_id = session.get("room") # User must be in a backend room
    if not room_id: print("Tried to connect frontend room - user is not in a backend room yet"); return False

    # Room must exist
    room = rooms.get(room_id)
    if not room: print("Tried to connect frontend room - backend room no longer exists"); return False

    return True

<h1>Blackjack Join and Leave Room</h1>

In [ ]:
@socketio.on("blackjack_player_join")
def blackjack_player_join():
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    game = room.game
    username = session.get("username")
    print("DEBUG username:", username)  # add this
    print("DEBUG session:", dict(session))  # add this
    blackjack_player = game.AddPlayer(request.sid, username)
    if blackjack_player is not None:                                  # check if successfully added the player - game might be full 
        sid_player_obj_mapping[request.sid] = blackjack_player        # globally accessible dictionary - not bound to a room

        # Tell other players to render this new player as a label
        socketio.emit("relay_player_info", {"id" : request.sid, "username": username,}, to=room_id)

        # Tell then newly connected player to render all previous players
        for player in game.players:
            socketio.emit("relay_player_info", {
                "id" : player.id,
                "username": player.username,
                "game_score" : player.game_score,
                "hand" : player.hand,
                "hand_total" : player.hand_total,
                "state" : player.state,
            }, to=request.sid)
        # As well as the dealer
        socketio.emit("relay_player_info", {
                "id" : "dealer",
                "hand" : game.dealer.hand,
                "hand_total" : game.dealer.hand_total,
            }, to=request.sid)

@socketio.on("blackjack_player_leave")
def blackjack_player_leave():
    if request.sid in sid_player_obj_mapping.keys():          # if sid exists in the mapping 
        player_obj = sid_player_obj_mapping[request.sid]
        room_id = session.get("room")
        room = rooms.get(room_id)
        game = room.game 
        if player_obj is not None: 
            del sid_player_obj_mapping[request.sid]
            game.RemovePlayer(player_obj)
        

<h1>Blackjack Hit and Stand</h1>

In [ ]:
@socketio.on("blackjack_hit_request")
def blackjack_hit_request():
    if not socket_validate(session): return
    
    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid

    room.game.PlayerHitRequest(sid)

@socketio.on("blackjack_stand_request")
def blackjack_stand_request():
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid

    room.game.PlayerStandRequest(sid)

<h1>Crazy Eight Join</h1>

In [ ]:
@socketio.on("crazyeights_player_join")
def crazyeights_player_join():
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    game = room.game
    username = session.get("username")
    print("DEBUG username:", username)  # add this
    print("DEBUG session:", dict(session))  # add this
    crazyeights_player = game.AddPlayer(request.sid, username)
    session["player_obj"] = crazyeights_player

    # Tell player the room size
    socketio.emit("crazyeights_room_size", {
        "room_size": room.num_players
    }, to=request.sid)

    # Tell everyone else in the room to render this new player
    socketio.emit("crazyeights_relay_player_info", {
        "id" : request.sid,
        "username": username,
        "hand": crazyeights_player.hand,
        "game_score": crazyeights_player.game_score
    }, to=room_id)

    # Tell this newly connected player about everyone who is already sitting at the table
    for player in game.players:
        socketio.emit("crazyeights_relay_player_info", {
            "id" : player.id,
            "username": player.username,
            "game_score" : player.game_score,
            "hand" : player.hand,
            "state" : player.state,
        }, to=request.sid)
        
    # If a round is already active, send them the current discard pile and active suit
    if game.discard_pile:
        socketio.emit("crazyeights_relay_board_info", {
            "top_card": game.discard_pile[-1]["code"],
            "current_suit": game.current_suit,
            "current_value": game.current_value
        }, to=request.sid)


<h1>Crazy Eight Game Functions</h1>

In [ ]:
@socketio.on("crazyeights_play_card")
def crazyeights_play_card(data):
    if not socket_validate(session): return
    
    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid
    card_code = data.get("card_code")

    room.game.PlayCardRequest(sid, card_code)

@socketio.on("crazyeights_choose_suit")
def crazyeights_choose_suit(data):
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid
    new_suit = data.get("suit")

    room.game.ChooseSuitRequest(sid, new_suit)

<h1>Socketio Connect and Disconnect</h1>

In [ ]:
@socketio.on("connect")
def socket_connect(*arg):
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid

    # Track connection
    socketio_connected[sid] = room

    # Join socketio room
    join_room(room_id)

    if room_id not in socketio_rooms:
        socketio_rooms.append(room_id)

    #
    emit("message", {"msg": f"Successfully joined a room - room_id: {room_id}"})

@socketio.on("disconnect")
def socket_disconnect(*arg):
    if not socket_validate(session): return

    room_id = session.get("room")
    room = rooms.get(room_id)
    sid = request.sid

    socketio_connected.pop(sid, None)

    leave_room(room_id)

    emit("message", {"msg": "Successfully left a room"})

<h1>MAIN</h1>

In [ ]:
def game_loop():
    last_time = time.time()
    TICK_RATE = 1/30  # 30 ticks/sec
    while True:
        current_time = time.time()
        dt = current_time - last_time
        last_time = current_time

        try:
            for room_id, room in rooms.items():
                room.game.Update(dt)  # pass delta time in seconds
        except Exception as e:
            print("Error in game loop:", e)
            traceback.print_exc()

        time.sleep(TICK_RATE)

if __name__ == '__main__': 
    # Added by Thomas McPhee
    # Start game loop
    thread = threading.Thread(target=game_loop, daemon=True)
    thread.start()


    app.run(host="0.0.0.0")